# EXACT Real API on Kaggle + Cloudflare Tunnel

Notebook này là bản **API thật** cho submit:

- Chạy Qwen3-8B + LoRA bằng `transformers + PEFT`.
- Type1 dùng parser `Final Answer` + verifier v35.
- Type2 dùng deterministic sanity solver.
- Tạo FastAPI `/predict` trong **cùng kernel** để endpoint dùng được model đang nằm trong RAM/GPU.
- Expose API ra public HTTPS bằng Cloudflare Tunnel.

Không cài vLLM trong notebook này.


In [ ]:
# CELL 0 — Config

RUN_SMOKE = True
SMOKE_N = 10
RUN_API = True
RUN_TUNNEL = True

BASE_MODEL = "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1"
TYPE1_LORA = "/kaggle/input/datasets/nguyenminhtric/exact-test/DATASET_DATA_TYPE_1"
TYPE1_ARTIFACT = "/kaggle/input/datasets/nguyenminhtric/exact-test/DATASET_DATA_TYPE_1/full_model_eval_v2_flatten_preds.json"
TYPE1_PATCH = "/kaggle/input/datasets/nguyenminhtric/exact-test/type1_fixed_patch/type1_fixed"

PORT = 9000

print("RUN_SMOKE:", RUN_SMOKE)
print("RUN_API:", RUN_API)
print("RUN_TUNNEL:", RUN_TUNNEL)
print("PORT:", PORT)


In [ ]:
# CELL 1 — Install dependencies, no vLLM here

!pip install -q transformers accelerate peft safetensors fastapi uvicorn requests httptools

import os, sys, json, re, time, gc, shutil, subprocess, threading
from pathlib import Path

import torch
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available(), "gpu_count:", torch.cuda.device_count())
!nvidia-smi


In [ ]:
# CELL 2 — Load artifact rows for smoke test

from pathlib import Path
import json

rows = json.loads(Path(TYPE1_ARTIFACT).read_text())
assert isinstance(rows, list)
print("rows:", len(rows))
print("row0 gold:", rows[0].get("gold"))
print("row0 prompt preview:")
print(rows[0]["prompt"][:800])


In [ ]:
# CELL 3 — Prepare verifier_v35 from patch

from pathlib import Path
import shutil, sys, types, importlib.util

WORK = Path("/kaggle/working/exact_runtime")
PATCH = Path(TYPE1_PATCH)
assert PATCH.exists(), PATCH

shutil.rmtree(WORK, ignore_errors=True)
(WORK / "app/type1_logic").mkdir(parents=True, exist_ok=True)

for p in [WORK / "app/__init__.py", WORK / "app/type1_logic/__init__.py"]:
    p.write_text("")

for name in ["parser.py", "prompt.py", "solver.py", "verifier_v35.py"]:
    src = PATCH / name
    dst = WORK / "app/type1_logic" / name
    assert src.exists(), src
    shutil.copy(src, dst)

def get_v35_module():
    if "app.type1_logic.verifier_v35" in sys.modules:
        return sys.modules["app.type1_logic.verifier_v35"]

    app_dir = WORK / "app"
    type1_dir = app_dir / "type1_logic"

    app_mod = types.ModuleType("app")
    app_mod.__path__ = [str(app_dir)]
    sys.modules["app"] = app_mod

    type1_mod = types.ModuleType("app.type1_logic")
    type1_mod.__path__ = [str(type1_dir)]
    sys.modules["app.type1_logic"] = type1_mod

    for name in ["parser", "prompt"]:
        path = type1_dir / f"{name}.py"
        spec = importlib.util.spec_from_file_location(f"app.type1_logic.{name}", path)
        mod = importlib.util.module_from_spec(spec)
        sys.modules[f"app.type1_logic.{name}"] = mod
        spec.loader.exec_module(mod)

    verifier = type1_dir / "verifier_v35.py"
    spec = importlib.util.spec_from_file_location("app.type1_logic.verifier_v35", verifier)
    v35 = importlib.util.module_from_spec(spec)
    sys.modules["app.type1_logic.verifier_v35"] = v35
    spec.loader.exec_module(v35)
    return v35

def call_v35(question, premises, model_answer):
    return get_v35_module().verify(question, premises, model_answer)

v35 = get_v35_module()
print("verifier loaded:", v35)
print("has verify:", hasattr(v35, "verify"))


In [ ]:
# CELL 4 — Load Qwen3-8B + LoRA with transformers/PEFT fp16

import gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(TYPE1_LORA, trust_remote_code=True)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base_model, TYPE1_LORA)
model.eval()

print("Loaded base + LoRA OK")
print("device map:", getattr(model, "hf_device_map", None))
!nvidia-smi


In [ ]:
# CELL 5 — Prediction helpers

import re, time, torch

def normalize_ans(ans):
    if ans is None:
        return None
    ans = str(ans).strip()
    low = ans.lower()
    if low == "uncertain":
        return "Unknown"
    if low in ["yes", "no", "unknown"]:
        return low.capitalize()
    if ans.upper() in ["A", "B", "C", "D"]:
        return ans.upper()
    return ans

def extract_final_answer(text):
    matches = re.findall(r"Final Answer:\s*([A-D]|Yes|No|Unknown|Uncertain)\b", text, flags=re.I)
    if not matches:
        return None
    return normalize_ans(matches[0])

def clean_until_final_answer(gen):
    kept = []
    for line in gen.splitlines():
        kept.append(line)
        if "Final Answer:" in line:
            break
    return "\n".join(kept).strip()

def generate_lora(prompt, max_new_tokens=128):
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    sec = time.time() - t0

    text = tokenizer.decode(out[0], skip_special_tokens=True)
    gen = text[len(prompt):] if text.startswith(prompt) else text
    clean = clean_until_final_answer(gen)
    raw_answer = extract_final_answer(clean)
    return clean, raw_answer, sec

def apply_verdict(verdict, current_ans):
    ans = current_ans
    premises_used = []
    note = None

    if isinstance(verdict, tuple):
        if len(verdict) >= 1 and verdict[0] is not None:
            ans = verdict[0]
        if len(verdict) >= 2:
            premises_used = verdict[1] or []
        if len(verdict) >= 3:
            note = verdict[2]
    elif isinstance(verdict, dict):
        candidate = verdict.get("answer") or verdict.get("pred") or verdict.get("label")
        if candidate is not None:
            ans = candidate
        premises_used = verdict.get("premises_used", [])
        note = verdict.get("reasoning") or verdict.get("proof") or verdict.get("certificate")
    elif isinstance(verdict, str):
        ans = verdict

    return normalize_ans(ans), premises_used, note

def predict_type1_row(row, max_new_tokens=128):
    clean, raw_ans, latency = generate_lora(row["prompt"], max_new_tokens=max_new_tokens)
    if raw_ans is None:
        raw_ans = "Unknown" if row.get("is_ynu", False) else None
    raw_ans = normalize_ans(raw_ans)

    try:
        verdict = call_v35(row["question"], row["premises_NL"], raw_ans)
    except Exception as e:
        verdict = (None, [], f"verifier_error:{type(e).__name__}:{str(e)[:120]}")

    final_ans, premises_used, note = apply_verdict(verdict, raw_ans)
    return {
        "answer": final_ans,
        "raw_answer": raw_ans,
        "latency_sec": latency,
        "clean_output": clean,
        "verdict": verdict,
        "premises_used": premises_used,
        "note": note,
    }


In [ ]:
# CELL 6 — Type1 smoke test

if RUN_SMOKE:
    ok = 0
    total = min(SMOKE_N, len(rows))
    for i, row in enumerate(rows[:total]):
        pred = predict_type1_row(row, max_new_tokens=128)
        gold = normalize_ans(row["gold"])
        correct = pred["answer"] == gold
        ok += int(correct)

        print(f"\n==== ROW {i} gold={gold} ====")
        print("pred:", pred["answer"], "gold:", gold, "correct:", correct, "latency:", round(pred["latency_sec"], 2))
        print("reasoning:", {"raw_answer": pred["raw_answer"], "verdict": pred["verdict"], "note": pred["note"]})
    print("\nPIPELINE SMOKE:", ok, "/", total)
else:
    print("RUN_SMOKE=False; skipped.")


In [ ]:
# CELL 7 — Type2 solver and live Type1 request function

import re

def _field(req, name, default=None):
    if isinstance(req, dict):
        return req.get(name, default)
    return getattr(req, name, default)

def solve_type2(req):
    query_id = _field(req, "query_id", "unknown")
    query = _field(req, "query", "") or ""
    q = query.lower()

    nums = re.findall(r"r\d*\s*=\s*([0-9.]+)\s*ohm", q)
    v = re.search(r"([0-9.]+)\s*v\b", q)

    if "parallel" in q and "current" in q and len(nums) >= 2 and v:
        rs = [float(x) for x in nums[:2]]
        V = float(v.group(1))
        I = sum(V / r for r in rs)
        ans = str(int(round(I))) if abs(I - round(I)) < 1e-9 else f"{I:.6g}"
        return {
            "query_id": query_id,
            "answer": ans,
            "unit": "A",
            "explanation": "Computed by deterministic formula solver.",
            "premises_used": [],
            "reasoning": {
                "type": "cot",
                "steps": [
                    "Compute branch currents with I=U/R.",
                    "Add branch currents.",
                    f"Result: {ans} A."
                ]
            },
        }

    return {
        "query_id": query_id,
        "answer": "0",
        "unit": "",
        "explanation": "The Type2 fallback could not solve this confidently.",
        "premises_used": [],
        "reasoning": None,
    }

def build_prompt_from_request(req):
    premises = list(_field(req, "premises", []) or [])
    query = _field(req, "query", "") or ""

    lines = [
        "You are solving a logic-based educational QA problem. Use only the given premises. Do not use outside knowledge.",
        "",
        "Premises:",
    ]
    for i, p in enumerate(premises, 1):
        lines.append(f"{i}. {p}")
    lines += [
        "",
        "Question:",
        query,
        "",
        "Reason step by step briefly, cite supporting premises if useful, and End with exactly one line: Final Answer: <Yes, No, or Unknown>",
    ]
    return "\n".join(lines) + "\n"

def predict_type1_live(req):
    query_id = _field(req, "query_id", "unknown")
    options = list(_field(req, "options", []) or [])
    premises = list(_field(req, "premises", []) or [])
    question = _field(req, "query", "") or ""

    clean, raw_ans, latency = generate_lora(build_prompt_from_request(req), max_new_tokens=128)
    if raw_ans is None:
        raw_ans = "Unknown"
    raw_ans = normalize_ans(raw_ans)

    try:
        verdict = call_v35(question, premises, raw_ans)
    except Exception as e:
        verdict = (None, [], f"verifier_error:{type(e).__name__}:{str(e)[:120]}")

    final_ans, premises_used, note = apply_verdict(verdict, raw_ans)

    if final_ans == "Unknown" and "Uncertain" in options:
        final_ans = "Uncertain"
    if options and final_ans not in options:
        final_ans = "Uncertain" if "Uncertain" in options else options[0]

    return {
        "query_id": query_id,
        "answer": final_ans,
        "unit": "",
        "explanation": "LoRA answer parsed from Final Answer, with v35 verifier override when proof is certain.",
        "premises_used": premises_used,
        "reasoning": {
            "raw_answer": raw_ans,
            "verdict": verdict,
            "note": note,
            "latency_sec": latency,
        },
    }

print("Type2 sanity:", solve_type2({
    "query_id": "T2_parallel",
    "query": "Two resistors R1 = 4 ohm and R2 = 6 ohm are in parallel across a 12 V battery. Find the total current."
}))


In [ ]:
# CELL 8 — Create FastAPI app inside this same kernel

from fastapi import FastAPI
from pydantic import BaseModel, Field
from typing import Literal
import uvicorn
import threading
import time
import requests

class PredictRequest(BaseModel):
    query_id: str
    type: Literal["type1", "type2"]
    query: str
    premises: list[str] = Field(default_factory=list)
    options: list[str] = Field(default_factory=list)

app = FastAPI(title="EXACT Real API")

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict")
def predict(req: PredictRequest):
    try:
        if req.type == "type2":
            return [solve_type2(req)]
        return [predict_type1_live(req)]
    except Exception as e:
        # Never 500 for valid requests.
        fallback = "Uncertain" if "Uncertain" in getattr(req, "options", []) else "Unknown"
        return [{
            "query_id": getattr(req, "query_id", "unknown"),
            "answer": fallback,
            "unit": "",
            "explanation": f"Fallback after internal error: {type(e).__name__}",
            "premises_used": [],
            "reasoning": None,
        }]

print("FastAPI app ready")


In [ ]:
# CELL 9 — Start local API server in a background thread

if RUN_API:
    # Stop old server object if this cell is rerun.
    if "server" in globals():
        try:
            server.should_exit = True
            time.sleep(2)
            print("old server signaled to stop")
        except Exception as e:
            print("old server stop error:", e)

    config = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="info")
    server = uvicorn.Server(config)
    server_thread = threading.Thread(target=server.run, daemon=True)
    server_thread.start()

    time.sleep(5)
    print("Local health:", requests.get(f"http://127.0.0.1:{PORT}/health", timeout=10).text)
else:
    print("RUN_API=False; skipped.")


In [ ]:
# CELL 10 — Test local /predict

LOCAL_PREDICT_URL = f"http://127.0.0.1:{PORT}/predict"

tests = [
    {
        "query_id": "T1_yes",
        "type": "type1",
        "query": "Does at least one student receive a scholarship?",
        "premises": ["Every student receives a scholarship."],
        "options": ["Yes", "No", "Uncertain"],
    },
    {
        "query_id": "T1_unknown",
        "type": "type1",
        "query": "Does at least one student publish a paper?",
        "premises": ["Every student receives a scholarship."],
        "options": ["Yes", "No", "Uncertain"],
    },
    {
        "query_id": "T2_parallel",
        "type": "type2",
        "query": "Two resistors R1 = 4 ohm and R2 = 6 ohm are in parallel across a 12 V battery. Find the total current.",
        "premises": [],
        "options": [],
    },
]

for payload in tests:
    r = requests.post(LOCAL_PREDICT_URL, json=payload, timeout=120)
    print("\n", payload["query_id"], r.status_code)
    print(r.text[:2000])


In [ ]:
# CELL 11 — Open Cloudflare Tunnel

if RUN_TUNNEL:
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared
    !chmod +x /kaggle/working/cloudflared

    import subprocess, time, pathlib, re

    # Stop old tunnel if any.
    subprocess.run("pkill -f cloudflared || true", shell=True)
    time.sleep(2)

    cf_log = "/kaggle/working/cloudflared_real_api.log"

    p_cf = subprocess.Popen(
        ["/kaggle/working/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
        stdout=open(cf_log, "w"),
        stderr=subprocess.STDOUT,
    )

    time.sleep(12)
    text = pathlib.Path(cf_log).read_text()
    print(text[-5000:])

    m = re.search(r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com", text)
    if not m:
        raise RuntimeError("Could not find trycloudflare URL. Inspect /kaggle/working/cloudflared_real_api.log")

    PUBLIC_BASE_URL = m.group(0)
    PUBLIC_PREDICT_URL = PUBLIC_BASE_URL + "/predict"
    PUBLIC_HEALTH_URL = PUBLIC_BASE_URL + "/health"

    print("PUBLIC_HEALTH_URL:", PUBLIC_HEALTH_URL)
    print("PUBLIC_PREDICT_URL:", PUBLIC_PREDICT_URL)
else:
    print("RUN_TUNNEL=False; skipped.")


In [ ]:
# CELL 12 — Test public /predict

import requests, json, time

print("PUBLIC_HEALTH_URL:", PUBLIC_HEALTH_URL)
print("PUBLIC_PREDICT_URL:", PUBLIC_PREDICT_URL)

print("health:", requests.get(PUBLIC_HEALTH_URL, timeout=30).text)

payload = {
    "query_id": "T2_public_test",
    "type": "type2",
    "query": "Two resistors R1 = 4 ohm and R2 = 6 ohm are in parallel across a 12 V battery. Find the total current.",
    "premises": [],
    "options": []
}

r = requests.post(PUBLIC_PREDICT_URL, json=payload, timeout=90)
print("T2 status:", r.status_code)
print(r.text)

payload = {
    "query_id": "T1_public_yes",
    "type": "type1",
    "query": "Does at least one student receive a scholarship?",
    "premises": ["Every student receives a scholarship."],
    "options": ["Yes", "No", "Uncertain"],
}

r = requests.post(PUBLIC_PREDICT_URL, json=payload, timeout=120)
print("T1 status:", r.status_code)
print(r.text[:2000])

print("\nSUBMIT THIS API URL:")
print(PUBLIC_PREDICT_URL)


In [ ]:
# CELL 13 — Stop server/tunnel after judging/testing

# Run this only when done.
# if "server" in globals():
#     server.should_exit = True
# subprocess.run("pkill -f cloudflared || true", shell=True)
# print("stopped")
